# 📘 KG course SPARQL notebook

A notebook to run SPARQL queries for the KG course at UM DACS.

1. Update the `g.parse()` calls in the first cell to import your RDF files.
2. In the same folder as the notebook create files with your SPARQL queries (e.g. `q1.rq`), and execute them with `run_query(g, 'q1.rq')`

Use the `.rq` file extension to get SPARQL syntax coloration

In [108]:
import sys
!pip install pandas oxrdflib Pygments rdflib

import pandas as pd
from IPython.display import display, HTML
from pygments import highlight
from pygments.lexers import SparqlLexer
from pygments.formatters import HtmlFormatter
from rdflib import Graph

In [109]:

def run_query(graph, query_path):
    try:
        with open(query_path, 'r') as file:
            query = file.read()
    except Exception as _e:
        print(f"No file for {query_path}")
        return
    results = graph.query(query)
    # Display the SPARQL query
    formatted_query = highlight(query, SparqlLexer(), HtmlFormatter(style='solarized-dark', full=True, nobackground=True))
    display(HTML(formatted_query))
    # Convert results to a Pandas DataFrame
    res_list = []
    for row in results:
        res_list.append([str(item) for item in row])
    df = pd.DataFrame(res_list, columns=[str(var) for var in results.vars]) if len(res_list) > 0 else pd.DataFrame()
    # Display the DataFrame as a table in Jupyter Notebook
    display(HTML(df.to_html()))

g = Graph(store="Oxigraph")

# TODO: modify/add paths to your RDF files
g.parse("./food_kg.ttl")
g.parse("./food_kg_ontology.ttl")
print(f"Working with {len(g)} triples")

Failed to convert Literal lexical form to value. Datatype=http://www.w3.org/2001/XMLSchema#duration, Converter=<function parse_xsd_duration at 0x000002DC7FAADA60>
Traceback (most recent call last):
  File "C:\Users\Pc\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\rdflib\term.py", line 2262, in _castLexicalToPython
    return conv_func(lexical)  # type: ignore[arg-type]
  File "C:\Users\Pc\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\rdflib\xsd_datetime.py", line 427, in parse_xsd_duration
    raise ValueError("Unable to parse duration string " + dur_string)
ValueError: Unable to parse duration string nan
Failed to convert Literal lexical form to value. Datatype=http://www.w3.org/2001/XMLSchema#duration, Converter=<function parse_xsd_duration at 0x000002DC7FAADA60>
Traceback (most recent call last):
  File "C:\Users\Pc\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\rdflib\term.py", line 2262, in _castLexicalToPython
    return conv_func(lexical)  # ty

Working with 1882 triples


1. Identify one type of quality check different than above, write and run SPARQL to implement the check and return the violating entities.

In [110]:
run_query(g, 'q1.rq')

,recipesWithMissingIngredients
0,20


Quality dimension: Completeness

The count shows how many recipes have undeclared ingredients. 0 indicates that all recipes are complete with respect to ingredients, and a higher value signals the lack of essential information.

2. Identify a second type of quality check different than above, write and run SPARQL to implement the check and return the violating entities.

In [111]:
run_query(g, 'q2.rq')

,recipe,name,num_Calories
0,http://kg-course/food-nutrition/recipe/42,Cabbage Soup,4


Quality dimension: Consistency

The recipes that are returned from this all have more than one distinct calorie value. Recipes having multiple calorie values might imply conflicting or duplicated nutritional data.

3. Identify a third type of quality check different than above, write and run SPARQL to implement the check and return the violating entities.

In [112]:
run_query(g, 'q3.rq')

,recipe,nutrition
0,http://kg-course/food-nutrition/recipe/48,cholesterol


Quality dimension: Accuracy

This query returns recipes with a schema:nutrition value that is a literal rather than a NutritionInformation resource. This is used to find any incorrect usage of the schema because nutrition should be modeled as an entity not a literal.

4. Identify a forth type of quality check different than above, write and run SPARQL to implement the check and return the violating entities.

In [113]:
run_query(g, 'q4.rq')

,recipe,time
0,http://kg-course/food-nutrition/recipe/74837,45


Quality dimension: Syntactic validity

This query shows recipes that have cookTime or prepTime values that are not typed as xsd:duration. These wrongly typed values violate the expected datatype and could be because of, for instance, incorrectly parsed time information (eg. "nan").

5. Identify a fifth type of quality check different than above, write and run SPARQL to implement the check and return the violating entities.

In [114]:
run_query(g, 'q5.rq')

,dimension,recipesWithInfo
0,How,0
1,When,55
2,Where,0
3,Who,0


Quality dimension: Provenance completeness

This query tells us how many recipes contain provenance data for that dimension, with greater values indicating better traceability or contextual understanding of the data.

6. Identify a sixth type of quality check different than above, write and run SPARQL to implement the check and return the violating entities.

In [115]:
run_query(g, 'q14.rq')

No file for q14.rq


Quality dimension: Consistency (range violation)

This query helps us find the recipes that have schema:nutrition properties pointing to an IRI that has not been typed as schema:NutritionInformation. Untyped or differently-typed nodes cannot be validated or reasoned over correctly. In the results, each row represents a recipe whose nutrition link points to a node with no declared class.

7. Identify a seventh type of quality check different than above, write and run SPARQL to implement the check and return the violating entities.

In [116]:
run_query(g, 'q15.rq')

No file for q15.rq


Quality dimension: Syntactic validity (datatype violation)

This query is for finding the recipes which have schema:prepTime values that are either not a literal at all or is a literal with a datatype other than xsd:duration. ISO 8601 duration is the only valid datatype for time properties in this ontology, and any deviation from this would prevent the value from being parsed, sorted, or compared properly.

8. Identify an eighth type of quality check different than above, write and run SPARQL to implement the check and return the violating entities.

In [117]:
run_query(g, 'interlinking.rq')

No file for interlinking.rq


9. Identify a ninth type of quality check different than above, write and run SPARQL to implement the check and return the violating entities.

Quality dimension: Consistency (schema coverage gap)

This query is used to find the properties that are actively used on schema:Recipe entities but are not declared anywhere in food_kg_ontology.ttl. These properties are therefore not owl:DatatypeProperty, owl:ObjectProperty, or rdf:Property. The occurrences count in the results show how frequently the associated undeclared property is used. A higher
count means that there is a bigger gapbetween the data and its schema.

In [118]:
run_query(g, 'q9.rq')

,recipesMissingInstructions
0,1


This query is used to spot if any recipes are missing instructions as this makes them useless for potential users and other aplications.

10. Identify a final type of quality check different than above, write and run SPARQL to implement the check and return the violating entities.

In [119]:
run_query(g, 'q10.rq')

,recipe,ingredient
0,http://kg-course/food-nutrition/recipe/88096,2 onions
1,http://kg-course/food-nutrition/recipe/88096,2 brown sugar
2,http://kg-course/food-nutrition/recipe/88096,2 beef flank steak
3,http://kg-course/food-nutrition/recipe/88096,2 Italian plum tomatoes
4,http://kg-course/food-nutrition/recipe/88096,1/2 fresh ground pepper
5,http://kg-course/food-nutrition/recipe/88096,1/2 - 3/4 tomato paste
6,http://kg-course/food-nutrition/recipe/88096,1 kosher salt
7,http://kg-course/food-nutrition/recipe/88096,1 coarse salt
8,http://kg-course/food-nutrition/recipe/88096,1 cabbage
9,http://kg-course/food-nutrition/recipe/74837,3 milk


This query was used to look through potential nonsensical ingredients as eg. 1/2 kosher salt